# Day 04 · How a machine learns a line

**Goal:** implement **gradient descent from scratch in NumPy** for simple linear regression, *watch* it learn (animated loss curve + line fit), feel what the **learning rate** does, see why **feature scaling** matters, compare **batch / mini-batch / stochastic** updates, then get the same line from the **closed form** and from **scikit-learn**.

**Run in Google Colab:** File → Open notebook → Upload → pick this `.ipynb`. Then *Runtime → Run all*.

**Runtime:** CPU is fine (everything runs in seconds).

| Section | What you'll do |
|---|---|
| 1 | The data: years of experience → salary (LPA) |
| 2 | Hypothesis + cost (MSE): score any line |
| 3 | Gradient descent from scratch + loss curve |
| 4 | 🎬 Animate the line learning |
| 5 | Learning-rate sweep: crawl, converge, explode |
| 6 | Feature scaling: the bowl becomes round (contour plot with the descent path) |
| 7 | Batch vs mini-batch vs stochastic |
| 8 | Closed form, `LinearRegression`, `SGDRegressor`, and a real dataset |
| 9 | 🎯 Try it yourself + Homework starter |

## 0. Setup
Everything is preinstalled in Colab.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import sklearn
np.random.seed(0)
sns.set_theme(style="whitegrid")
print("numpy", np.__version__, "| sklearn", sklearn.__version__)

## 1. The data: "what salary should I ask for?" 💼

Thirty (experience, salary) points from friends and seniors. The true relationship we *pretend* the market follows is `salary = 3 + 1.8 × years + noise` — the machine does **not** know these numbers; it has to find them from the dots.

In [ ]:
rng = np.random.default_rng(4)
x = np.round(rng.uniform(0, 10, 30), 1)                    # years of experience
y = np.round(3 + 1.8 * x + rng.normal(0, 1.2, 30), 2)      # salary in LPA
m = len(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color="tab:blue")
ax.set(xlabel="years of experience", ylabel="salary (LPA)", title="30 data points. Which line?")
plt.show()
print("x[:5] =", x[:5], "\ny[:5] =", y[:5])

## 2. Hypothesis and cost: score any line 🎯

**Hypothesis:** `ŷ = w·x + b`. The hypothesis space is *all straight lines*; `w` and `b` are the only numbers the machine may change.

**Cost (MSE):** `J(w, b) = 1/(2m) · Σ (ŷᵢ − yᵢ)²`.
**Analogy:** Zomato says "32 min", the food arrives in 40 → error 8; another arrives 5 early → error −5. Adding raw errors lets +8 and −5 cancel, so we *square* them (big misses hurt more), then average. The `/2` just cancels the 2 from differentiating a square.

In [ ]:
def predict(x, w, b):
    return w * x + b                       # vectorised: works on the whole array

def cost(x, y, w, b):
    err = predict(x, w, b) - y
    return (err ** 2).mean() / 2           # J = 1/(2m) Σ (ŷ - y)²

for w, b in [(0, 0), (1, 5), (1.8, 3), (2.5, 0)]:
    print(f"line ŷ = {w}x + {b:<3}  ->  J = {cost(x, y, w, b):7.3f}")

# Three candidate lines drawn on the data
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color="tab:blue")
xs = np.linspace(0, 10, 2)
for w, b, c in [(0, 0, "gray"), (1, 5, "tab:orange"), (1.8, 3, "tab:green")]:
    ax.plot(xs, predict(xs, w, b), color=c, label=f"w={w}, b={b}, J={cost(x, y, w, b):.2f}")
ax.legend(); ax.set(xlabel="years", ylabel="LPA", title="Lower J = better line"); plt.show()

## 3. Gradient descent from scratch ⛰️

**Analogy:** fog on a hillside. You can't see the valley but you can feel the slope under your feet. Step downhill, feel again, repeat. The **gradient** is the slope, the **learning rate α** is your stride.

The two gradients (derive them once by hand — it's a 3-line chain rule):

- `∂J/∂w = 1/m · Σ (ŷᵢ − yᵢ) · xᵢ`  → `(err * x).mean()`
- `∂J/∂b = 1/m · Σ (ŷᵢ − yᵢ)`      → `err.mean()`

Update both **from the old values**: `w -= α·dw`, `b -= α·db`.

In [ ]:
def gradient_descent(x, y, lr=0.01, epochs=200, w0=0.0, b0=0.0):
    """Batch gradient descent for ŷ = w x + b. Returns w, b and the per-epoch history."""
    w, b = w0, b0
    hist = {"J": [], "w": [], "b": []}
    for epoch in range(epochs):
        err = predict(x, w, b) - y         # (ŷ - y) for all m points, shape (m,)
        dw = (err * x).mean()              # ∂J/∂w
        db = err.mean()                    # ∂J/∂b
        w -= lr * dw                       # step against the gradient
        b -= lr * db                       # (dw, db computed from the OLD w, b: correct)
        hist["J"].append(cost(x, y, w, b)); hist["w"].append(w); hist["b"].append(b)
    return w, b, hist

w, b, hist = gradient_descent(x, y, lr=0.01, epochs=300)
print(f"after 300 epochs: w = {w:.3f}, b = {b:.3f}, J = {hist['J'][-1]:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(x, y); axes[0].plot(xs, predict(xs, w, b), color="tab:orange", lw=2)
axes[0].set(title=f"learned line: ŷ = {w:.2f}x + {b:.2f}", xlabel="years", ylabel="LPA")
axes[1].plot(hist["J"]); axes[1].set(title="loss curve (the ECG of training)", xlabel="epoch", ylabel="J", yscale="log")
plt.show()

> 🧠 **Interview Q:** *"How does gradient descent work?"* Start with random parameters → compute the cost → compute the gradient of the cost w.r.t. each parameter → move each parameter a small step (α) in the opposite direction → repeat until the cost stops improving.

## 4. 🎬 Animate the line learning
Each frame = one epoch (we show every 3rd of the first 150). Press ▶ on the player. Left: the line moving; right: the loss dropping.

In [ ]:
w_a, b_a, h = gradient_descent(x, y, lr=0.01, epochs=150)
frames = list(range(0, 150, 3))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(x, y, color="tab:blue"); (line,) = ax1.plot([], [], color="tab:orange", lw=2.5)
ax1.set(xlim=(0, 10.5), ylim=(0, 24), xlabel="years", ylabel="LPA"); title = ax1.set_title("")
(curve,) = ax2.plot([], [], color="tab:red"); ax2.set(xlim=(0, 150), ylim=(0.3, 120), yscale="log", xlabel="epoch", ylabel="J")
plt.close(fig)                                             # don't show the static figure

def update(k):
    w_k, b_k = h["w"][k], h["b"][k]
    line.set_data(xs, predict(xs, w_k, b_k))
    curve.set_data(range(k + 1), h["J"][:k + 1])
    title.set_text(f"epoch {k:3d}   w={w_k:.2f}  b={b_k:.2f}  J={h['J'][k]:.2f}")
    return line, curve, title

anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
HTML(anim.to_jshtml())

## 5. Learning-rate sweep: crawl, converge, explode 🎚️

α trades speed for stability. For this data (x from 0 to 10) the stability limit is about **α < 0.045** (it's `2 / λ_max` of the Hessian — you'll meet that on Day 20). Watch what happens on either side.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for lr in [0.001, 0.003, 0.01, 0.03, 0.05]:
    _, _, h = gradient_descent(x, y, lr=lr, epochs=100)
    J = np.clip(h["J"], 1e-3, 1e6)                          # clip so the diverging run stays on the chart
    ax.plot(J, label=f"α = {lr}" + ("   ← diverges!" if h["J"][-1] > 1e3 else ""))
ax.set(yscale="log", xlabel="epoch", ylabel="J (log)", title="Same data, five learning rates")
ax.legend(); fig.savefig("lr_sweep.png", dpi=150, bbox_inches="tight"); plt.show()

for lr in [0.001, 0.01, 0.03, 0.05]:
    _, _, h = gradient_descent(x, y, lr=lr, epochs=100)
    print(f"α={lr:<6} J after 100 epochs = {h['J'][-1]:.4g}")

## 6. Feature scaling: the bowl becomes round 🥣

With two parameters the cost is a bowl in 3-D. Because x ranges 0–10 while b's "direction" is scale 1, the bowl is a **long narrow valley**: steep across, flat along. One α can't suit both directions.

**Standardise** `x' = (x − mean)/std` → the bowl becomes round → any sensible α works and convergence is 10–100× faster. This is why every sklearn pipeline starts with `StandardScaler`.

In [ ]:
x_s = (x - x.mean()) / x.std()             # mean 0, std 1

w_s, b_s, h_s = gradient_descent(x_s, y, lr=0.5, epochs=30)      # 50x larger α, 10x fewer epochs
print(f"scaled  : α=0.5, 30 epochs -> J = {h_s['J'][-1]:.4f}  (w'={w_s:.3f}, b'={b_s:.3f})")
_, _, h_r = gradient_descent(x, y, lr=0.01, epochs=30)
print(f"raw     : α=0.01, 30 epochs -> J = {h_r['J'][-1]:.4f}")

# Convert the scaled-space line back to raw units: ŷ = w' (x-μ)/σ + b'  =  (w'/σ) x + (b' - w' μ/σ)
w_back, b_back = w_s / x.std(), b_s - w_s * x.mean() / x.std()
print(f"back in raw units: w = {w_back:.3f}, b = {b_back:.3f}")

In [ ]:
# The picture that explains it: contour plots of J with the descent path, raw vs scaled
def contour_with_path(ax, xx, lr, epochs, title, wspan=3):
    w_opt = ((xx - xx.mean()) * (y - y.mean())).sum() / ((xx - xx.mean()) ** 2).sum()
    b_opt = y.mean() - w_opt * xx.mean()
    W = np.linspace(w_opt - wspan, w_opt + wspan, 120); B = np.linspace(b_opt - 12, b_opt + 12, 120)
    WW, BB = np.meshgrid(W, B)
    JJ = ((WW[..., None] * xx + BB[..., None] - y) ** 2).mean(axis=-1) / 2     # broadcasting: (120,120,30)
    ax.contour(WW, BB, np.log10(JJ), levels=25, cmap="viridis")
    _, _, h = gradient_descent(xx, y, lr=lr, epochs=epochs)
    ax.plot([0] + h["w"], [0] + h["b"], "o-", color="tab:red", ms=3, lw=1, label=f"GD path (α={lr}, {epochs} ep)")
    ax.plot(w_opt, b_opt, "*", color="gold", ms=15, mec="k", label="minimum")
    ax.set(xlabel="w", ylabel="b", title=title); ax.legend(loc="upper right", fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
contour_with_path(axes[0], x,   lr=0.01, epochs=150, title="raw x: narrow valley -> zig-zag, slow")
contour_with_path(axes[1], x_s, lr=0.5,  epochs=15,  title="standardised x: round bowl -> straight to the bottom", wspan=12)
axes[1].set_aspect("equal")
plt.show()

## 7. Batch vs mini-batch vs stochastic 🔀

| Variant | examples per update | updates per epoch | path |
|---|---|---|---|
| **Batch** | all m | 1 | smooth, exact gradient |
| **Stochastic (SGD)** | 1 | m | noisy zig-zag, cheap steps |
| **Mini-batch** | 32–512 | m / batch | best of both; what every neural net uses |

We use the standardised x so all three can share α.

In [ ]:
def gd_variant(x, y, lr, epochs, batch_size, seed=0):
    """batch_size = len(x) -> batch GD; 1 -> SGD; anything between -> mini-batch."""
    rng = np.random.default_rng(seed)
    w, b, m = 0.0, 0.0, len(x)
    J_hist, path = [], [(w, b)]
    for _ in range(epochs):
        idx = rng.permutation(m)                          # shuffle every epoch
        for s in range(0, m, batch_size):
            i = idx[s:s + batch_size]
            err = predict(x[i], w, b) - y[i]
            w -= lr * (err * x[i]).mean()
            b -= lr * err.mean()
            path.append((w, b))
        J_hist.append(cost(x, y, w, b))
    return w, b, J_hist, np.array(path)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for bs, name in [(30, "batch"), (5, "mini-batch (5)"), (1, "stochastic")]:
    w_v, b_v, J_v, path = gd_variant(x_s, y, lr=0.1, epochs=20, batch_size=bs)
    axes[0].plot(J_v, marker="o", ms=3, label=f"{name}: J={J_v[-1]:.3f}")
    axes[1].plot(path[:, 0], path[:, 1], "-", lw=1, alpha=.8, label=f"{name} ({len(path)-1} updates)")
axes[0].set(title="loss per epoch (α=0.1, scaled x)", xlabel="epoch", ylabel="J", yscale="log"); axes[0].legend()
axes[1].plot(w_s, b_s, "*", color="gold", ms=15, mec="k"); axes[1].set(title="path in (w, b) space", xlabel="w'", ylabel="b'"); axes[1].legend(fontsize=8)
plt.show()

## 8. Closed form, scikit-learn, and a real dataset 📐

For a *straight line* there is an exact answer (the **normal equation**): `w = cov(x, y) / var(x)`, `b = ȳ − w·x̄`. `sklearn.LinearRegression` uses the matrix version (least squares via SVD) — no iterations. Gradient descent earns its keep when there is no closed form (neural nets) or the data is too big for one.

In [ ]:
# 1) closed form
w_cf = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b_cf = y.mean() - w_cf * x.mean()

# 2) sklearn LinearRegression (what you'll use at work)
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X = x.reshape(-1, 1)                                     # sklearn wants (n_samples, n_features)
lin = LinearRegression().fit(X, y)

# 3) sklearn's gradient descent, with scaling built into a pipeline
sgd = make_pipeline(StandardScaler(),
                    SGDRegressor(learning_rate="constant", eta0=0.01, max_iter=1000, tol=1e-6, random_state=0)).fit(X, y)

# 4) our own GD, run long enough
w_gd, b_gd, _ = gradient_descent(x, y, lr=0.03, epochs=3000)

print(f"{'method':<22}{'w':>8}{'b':>8}{'pred(2 yrs)':>14}")
print(f"{'closed form':<22}{w_cf:8.3f}{b_cf:8.3f}{predict(2.0, w_cf, b_cf):14.2f}")
print(f"{'LinearRegression':<22}{lin.coef_[0]:8.3f}{lin.intercept_:8.3f}{lin.predict([[2.0]])[0]:14.2f}")
print(f"{'SGDRegressor':<22}{'-':>8}{'-':>8}{sgd.predict([[2.0]])[0]:14.2f}   (scaled coefs live inside the pipeline)")
print(f"{'our GD (3000 ep)':<22}{w_gd:8.3f}{b_gd:8.3f}{predict(2.0, w_gd, b_gd):14.2f}")
print(f"\nR² on training data: {lin.score(X, y):.3f}   (Day 12 explains R²)")

### A real dataset: restaurant bill → tip
Same two lines of sklearn on real data (`tips`, 244 restaurant bills). The line tells you the "tip rate".

In [ ]:
try:
    tips = sns.load_dataset("tips")
except Exception:
    tips = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv")

Xt, yt = tips[["total_bill"]].values, tips["tip"].values
lin_t = LinearRegression().fit(Xt, yt)
print(f"tip ≈ {lin_t.coef_[0]:.3f} × bill + {lin_t.intercept_:.2f}   -> people tip about {lin_t.coef_[0]:.0%} of the bill + a fixed ${lin_t.intercept_:.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(tips, x="total_bill", y="tip", hue="time", ax=ax)
bills = np.linspace(0, 55, 2).reshape(-1, 1)
ax.plot(bills, lin_t.predict(bills), color="tab:red", lw=2, label="LinearRegression")
ax.legend(); ax.set(title="tips dataset: one line, learned from data"); plt.show()

## 9. 🎯 Try it yourself (in class, 5 minutes)

Add **early stopping** to `gradient_descent`: stop when the improvement in J between epochs is smaller than `tol` (say `1e-6`). Return the number of epochs actually run. Then report, for the *scaled* x, how many epochs α = 0.05, 0.1, 0.5, 1.0 need to converge.

In [ ]:
def gradient_descent_es(x, y, lr=0.01, max_epochs=5000, tol=1e-6):
    w, b = 0.0, 0.0
    prev = cost(x, y, w, b)
    for epoch in range(1, max_epochs + 1):
        # YOUR CODE HERE: one GD step (dw, db, update), then compute J
        # if prev - J < tol: return w, b, epoch
        # prev = J
        pass
    return w, b, max_epochs

for lr in [0.05, 0.1, 0.5, 1.0]:
    w_e, b_e, n = gradient_descent_es(x_s, y, lr=lr)
    print(f"α={lr:<5} converged in {n} epochs  (w'={w_e:.3f}, b'={b_e:.3f})")

<details><summary>Solution</summary>

```python
def gradient_descent_es(x, y, lr=0.01, max_epochs=5000, tol=1e-6):
    w, b = 0.0, 0.0
    prev = cost(x, y, w, b)
    for epoch in range(1, max_epochs + 1):
        err = predict(x, w, b) - y
        w -= lr * (err * x).mean()
        b -= lr * err.mean()
        J = cost(x, y, w, b)
        if prev - J < tol:
            return w, b, epoch
        prev = J
    return w, b, max_epochs

# α=0.05 -> 159 epochs, α=0.1 -> 81, α=0.5 -> 15, α=1.0 -> 2 (for a perfectly round bowl α=1 lands exactly at the bottom in one step)
```
</details>

## Homework starter 🏠

- **Easy:** learning-rate sweep chart (`lr_sweep.png` is already saved above) — write one sentence per curve, post it on LinkedIn with your repo link.
- **Medium (daily life):** predict your commute — 15 trips of (km, minutes); fit with your GD *and* sklearn; interpret w (min/km) and b (waiting time?).
- **Stretch:** two features `ŷ = w₁x₁ + w₂x₂ + b` with `X @ w`; show divergence without scaling and convergence with `StandardScaler`.

Scaffold for Medium (replace the fake trips with yours):

In [ ]:
# --- Medium: your commute ---
trips = pd.DataFrame({
    "km":      [2.1, 4.5, 3.2, 8.0, 5.5, 1.2, 6.8, 4.0, 9.5, 3.7, 7.2, 2.8, 5.0, 6.1, 4.4],   # replace with yours
    "minutes": [12, 22, 18, 41, 27, 9, 35, 21, 48, 19, 36, 15, 25, 31, 23],
})
xk, ym = trips["km"].values, trips["minutes"].values
w_c, b_c, _ = gradient_descent(xk, ym, lr=0.01, epochs=3000)
lin_c = LinearRegression().fit(trips[["km"]], ym)
print(f"our GD : minutes ≈ {w_c:.2f} × km + {b_c:.2f}")
print(f"sklearn: minutes ≈ {lin_c.coef_[0]:.2f} × km + {lin_c.intercept_:.2f}")
print(f"-> about {lin_c.coef_[0]:.1f} min per km, plus ~{lin_c.intercept_:.0f} min fixed (waiting / walking)")
# YOUR CODE HERE: replace the numbers with your real trips, plot the points + line, commit the notebook.

In [ ]:
# --- Stretch scaffold: two features, vectorised with a weight vector ---
x2 = rng.uniform(0, 1000, 30)                         # "monthly hours" on a wildly different scale
X2 = np.column_stack([x, x2])                         # (30, 2)
y2 = 3 + 1.8 * x + 0.004 * x2 + rng.normal(0, 1.2, 30)

def gd_multi(X, y, lr, epochs):
    w = np.zeros(X.shape[1]); b = 0.0
    for _ in range(epochs):
        err = X @ w + b - y                           # (30,)
        w -= lr * (X.T @ err) / len(y)                # gradient for every weight at once: (2,)
        b -= lr * err.mean()
    return w, b, ((X @ w + b - y) ** 2).mean() / 2

with np.errstate(all="ignore"):                       # we EXPECT overflow here
    J_raw = gd_multi(X2, y2, lr=1e-3, epochs=500)[2]
print("raw    :", "DIVERGED (nan/inf)" if not np.isfinite(J_raw) else J_raw, "<- x2 ~ 1000 makes the valley extremely narrow; try lr=1e-6 and watch w1 barely move")
Xs2 = (X2 - X2.mean(axis=0)) / X2.std(axis=0)        # broadcasting from Day 03
print("scaled :", gd_multi(Xs2, y2, lr=0.5, epochs=100)[2], "<- converged")

## Key takeaways 🔑

- **Learning = adjust parameters to reduce a measured cost.** Model (`w`, `b`) ≠ cost (MSE) ≠ optimiser (gradient descent).
- **Gradients for MSE:** `dw = mean(err · x)`, `db = mean(err)`; update both from the old values with `w -= α·dw`.
- **Learning rate:** too small → crawl; too large → oscillate/diverge (NaN). Sweep on a log scale; read the loss curve.
- **Feature scaling** turns the narrow valley into a round bowl → larger α is stable, convergence is 10–100× faster. `StandardScaler` first, always. Scale new inputs with the *training* mean/std.
- **Batch / mini-batch / SGD** differ in how many examples per update; mini-batch is the deep-learning default.
- **For a line, the closed form exists** (`LinearRegression` uses it). GD is for when it doesn't (Day 19) or data is huge.

**Before Day 05:** post your `lr_sweep.png` + 4 lines on LinkedIn; tomorrow: many features, curves, overfitting, Ridge/Lasso.